In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [4]:
from langchain_community.document_loaders import PyPDFLoader, MergedDataLoader

In [5]:
docs = MergedDataLoader([
    PyPDFLoader("data/GTB_gold_Nov23.pdf"),
    PyPDFLoader("data/GTB_platinum_Nov23.pdf")
]).load()

In [7]:
len(docs)

53

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

slices = splitter.split_documents(docs)

In [12]:
len(slices)

200

In [13]:
from langchain_openai.embeddings import OpenAIEmbeddings

In [14]:
embedding_model = OpenAIEmbeddings()

embedding_model.model

'text-embedding-ada-002'

In [15]:
from langchain_community.vectorstores import InMemoryVectorStore

In [16]:
vectorstore = InMemoryVectorStore.from_documents(
    documents=slices,
    embedding=embedding_model
)

In [32]:
retriever = vectorstore.as_retriever(kwargs={"k": 2})

In [19]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [29]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """Gere uma consulta de pesquisa para um banco de dados vetorial a partir
     de uma pergunta do usuario, permitindo uma resposta mais precisa para a busca semantica

     Gere somente o texto da consulta e nada de texto adicional
     """),
    ("human", "{query}")
])

query_model = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.5,
    api_key=openai_api_key
)

rewrite_chain = rewrite_prompt | query_model | StrOutputParser()

In [30]:
query = "Como lidar com cartão roubado?"

In [31]:
rewrite_chain.invoke({"query": query})

'{"query": "Procedimentos e dicas para lidar com cartão de crédito ou débito roubado, incluindo bloqueio, denúncia às autoridades e prevenção de fraudes"}'

In [34]:
from langchain_core.runnables import RunnablePassthrough

In [33]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Responda utilizando exclusivamente o conteúdo fornecido: \n\nContexto: \n{contexto}"),
    ("human", "{query}")
])

In [40]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [41]:
rewrite_rag_chain = (
    {
        "contexto": RunnablePassthrough() | rewrite_chain | retriever | format_docs,
        "query": RunnablePassthrough()
    } | rag_prompt | query_model | StrOutputParser()
)

In [45]:
rewrite_rag_chain.invoke(query)

'De acordo com as informações fornecidas, em caso de perda ou roubo do seu cartão Mastercard Gold™, você deve entrar em contato com os Serviços de Assistência de Viagem. Eles fornecerão assistência para a substituição do cartão, além de ajudar na comunicação com a polícia local, consulados, companhias aéreas e outras entidades apropriadas.'